# Harmony4D Drive Benchmark → Google Sheets (Colab)

Chạy `run_drive_benchmark.py` trên Google Colab và xuất kết quả trực tiếp ra **Google Sheets** (không xuất CSV lên Drive):
- **Tự động ghim dòng tiêu đề** (`freeze(rows=1)`) + định dạng bảng qua `brute_force_runner.decorate`.
- **Tổ chức gọn trên Google Drive**: tự tạo 1 thư mục gốc (`OUTPUT_DRIVE_FOLDER`), bên trong **mỗi sequence là 1 thư mục riêng**, và **mỗi người là 1 file Google Sheet riêng** gồm 3 tab: `<method>` (so với baseline), `naive` (khi bật `COMPARE_WITH_NAIVE`), và `compare` (ghép từng cặp so **`method` vs `baseline`** & **`method` vs `naive`**).
- **Chấm MPJPE / Accel 0% GT** bằng extrinsics COLMAP gốc của Harmony4D (`configs/harmony4d_extrinsics.json` — 39/39 sequence tập test, hoặc `colmap/workplace/images.txt` trên Drive nếu có).

In [ ]:
#@title 1. Cài môi trường, clone repo, tải SMPL_NEUTRAL.pkl và cấu hình Multi-Person
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
BRANCH = "experiment/observability-aware-trust-region"  #@param {type:"string"}
SMPL_NEUTRAL_FILE_ID = "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"  #@param {type:"string"}
FUSION_METHOD = "partwise"  #@param ["partwise", "aotr", "naive", "adaptive_naive"]
COMPARE_WITH_NAIVE = True  #@param {type:"boolean"}
MULTI_PERSON = True  #@param {type:"boolean"}
STATIC_CAMERA = False  #@param {type:"boolean"}
DRIVE_DATASET_ROOT = "/content/drive/MyDrive/Harmony4D/test"  #@param {type:"string"}
CONFIGS = ""  #@param {type:"string"}
REVERSE = True  #@param {type:"boolean"}
WORKERS = 0  #@param {type:"integer"}
OUTPUT_DRIVE_FOLDER = ""  #@param {type:"string"}
SHEET_PREFIX = ""  #@param {type:"string"}
DRIVE_FOLDER_ID = ""  #@param {type:"string"}
PUBLIC_READ = True  #@param {type:"boolean"}

REPO_DIR = Path("/content/optimization_model_monocular_3")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-f", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "chumpy"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "PyYAML", "gspread", "numpy<2.3"], check=True)

smpl_path = REPO_DIR / "models" / "SMPL_NEUTRAL.pkl"
if not smpl_path.exists() or smpl_path.stat().st_size < 1_000_000:
    import gdown
    smpl_path.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=SMPL_NEUTRAL_FILE_ID, output=str(smpl_path), quiet=False)

import yaml
from fusion_pipeline.aotr_benchmark import report_csv_name

def apply_method_config(method_name):
    cfg_path = REPO_DIR / "configs" / "pipeline.yml"
    cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    cfg["fusion"]["method"] = method_name
    cfg["fusion"].setdefault("occlusion", {})["multi_person"] = bool(MULTI_PERSON)
    cfg["fusion"].setdefault("partwise", {}).setdefault("multi_person", {})["enabled"] = bool(MULTI_PERSON)
    cfg["fusion"]["partwise"]["multi_person"]["static_camera_orientation"] = bool(MULTI_PERSON and STATIC_CAMERA)
    cfg["fusion"].setdefault("aotr", {})["temporal_alignment_fallback"] = bool(MULTI_PERSON)
    cfg["fusion"]["aotr"]["max_cross_view_bone_angle_deg"] = 50.0 if MULTI_PERSON else None
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
    rname = report_csv_name(cfg)
    return rname, Path(rname).stem.replace("brute_force_report_", "")

METHODS = [FUSION_METHOD] + (["naive"] if COMPARE_WITH_NAIVE and FUSION_METHOD != "naive" else [])
METHOD_REPORTS = {m: apply_method_config(m) for m in reversed(METHODS)}
REPORT_NAME, REPORT_TAG = METHOD_REPORTS[FUSION_METHOD]
ext_db = REPO_DIR / "configs" / "harmony4d_extrinsics.json"
n_ext = len(__import__("json").loads(ext_db.read_text(encoding="utf-8")).get("sequences", {})) if ext_db.is_file() else 0
print(f"Ready | branch={BRANCH} | methods={METHODS} | multi_person={MULTI_PERSON} | static_cam={STATIC_CAMERA} | extrinsics_0gt={n_ext} seqs | tag={REPORT_TAG} | cpus={os.cpu_count()}")

In [ ]:
#@title 2. Chạy benchmark & ghi ra Google Drive (1 thư mục gốc / mỗi sequence 1 folder / mỗi người 1 Google Sheet)
import csv, os, shutil, subprocess, sys, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import gspread, yaml
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build
from brute_force_runner import decorate
from run_drive_benchmark import is_drive_segment

auth.authenticate_user()
creds, _ = default()
GC = gspread.authorize(creds)
DRIVE_SVC = build("drive", "v3", credentials=creds, cache_discovery=False)
DRIVE_ROOT = Path(DRIVE_DATASET_ROOT.strip()) if DRIVE_DATASET_ROOT.strip() else None
if DRIVE_ROOT is not None:
    drive.mount("/content/drive", force_remount=False)
    if not DRIVE_ROOT.exists():
        # ponytail: handle leading space in Drive folder name (' Harmony4D') or nested shortcut
        mydrive = Path("/content/drive/MyDrive")
        found = []
        for pat in ("*", "*/*"):
            for p in mydrive.glob(pat):
                if p.is_dir() and p.name.strip().lower() == "harmony4d":
                    found.append(p / "test" if (p / "test").is_dir() else p)
        if found:
            DRIVE_ROOT = found[0]
            print(f"[DriveMount] Tự tìm thấy thư mục Drive: {DRIVE_ROOT}", flush=True)
        else:
            top = sorted(p.name for p in mydrive.iterdir())[:20] if mydrive.exists() else []
            raise FileNotFoundError(f"Không thấy {DRIVE_ROOT}. Các thư mục trong MyDrive: {top}")
SHEET_URLS = {}
_LAST_SYNCED = {}

def _stage_file(src, dst, copy=False):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if copy:
        # ponytail: copy hot pkl/pth/zip once to local /content SSD to avoid 1000+ Drive FUSE reads
        if dst.is_file() and not dst.is_symlink() and dst.stat().st_size == src.stat().st_size:
            return
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        shutil.copy2(src, dst)
    else:
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        dst.symlink_to(src)

def stage_from_mounted_drive(cfg_file, seq_name):
    """Copy hot pkl/pth/zip/colmap once to local SSD & symlink videos (0 gdown, 0 Drive FUSE bottleneck)."""
    matches = [p for pat in (seq_name, f"*/{seq_name}", f"*/*/{seq_name}") for p in DRIVE_ROOT.glob(pat) if p.is_dir() and (p / "exo").is_dir()]
    if not matches:
        return False
    src_seq = matches[0]
    dst_seq = REPO_DIR / "input" / "drive_benchmark" / seq_name
    dst_seq.mkdir(parents=True, exist_ok=True)
    gt_zip = src_seq / "processed_data" / "poses3d.zip"
    dst_zip = dst_seq / "poses3d.zip"
    copy_jobs = []
    if gt_zip.is_file():
        copy_jobs.append((gt_zip, dst_zip))
    elif (src_seq / "processed_data" / "poses3d").is_dir() and not dst_zip.exists():
        with zipfile.ZipFile(dst_zip, "w") as zf:
            for npy in sorted((src_seq / "processed_data" / "poses3d").glob("*.npy")):
                zf.write(npy, f"poses3d/{npy.name}")
    else:
        return False
    for rel in ("colmap/workplace/images.txt", "colmap/workplace/cameras.txt",
                "colmap/workplace/colmap_from_aria_transforms.pkl", "colmap/workplace/aria_from_colmap_transforms.pkl",
                "colmap/workplace/scale.npy", "colmap/sparse/0/images.txt"):
        if (src_seq / rel).is_file():
            copy_jobs.append((src_seq / rel, dst_seq / rel))
    seg = (yaml.safe_load(cfg_file.read_text(encoding="utf-8")) or {})["segments"][0]
    for cam in seg.get("cameras", []):
        cid = cam["id"]
        src_cam = src_seq / "exo" / cid
        pkls = sorted(src_cam.glob("*wham*/wham_*.pkl")) or sorted(src_cam.glob("*/wham_*.pkl"))
        vids = sorted(src_cam.glob("images*/*.mp4"))
        if not pkls or not vids:
            return False
        stem = f"harmony4D_{seq_name}_{cid}"
        dst_wham = dst_seq / "exo" / cid / f"{stem}_wham"
        copy_jobs.append((pkls[0], dst_wham / f"wham_{stem}_ffmpeg.pkl"))
        trks = sorted(pkls[0].parent.glob("tracking_results*.pth"))
        if trks:
            copy_jobs.append((trks[0], dst_wham / "tracking_results.pth"))
        _stage_file(vids[0], dst_seq / "exo" / cid / "images" / f"{stem}_ffmpeg.mp4", copy=False)
    with ThreadPoolExecutor(max_workers=4) as pool:
        list(pool.map(lambda pair: _stage_file(pair[0], pair[1], copy=True), copy_jobs))
    print(f"[DriveMount] Đã copy {len(copy_jobs)} file pkl/pth/zip {seq_name} về SSD local (symlink mp4)", flush=True)
    return True

def sheet_title(seq_name, person):
    return f"{SHEET_PREFIX}{seq_name}_{person}_{REPORT_TAG}"

_SHEETS = {}
_FOLDERS = {}
def _gs_retry(fn, *args, **kwargs):
    # ponytail: retry transient Drive/Sheets 429 & 5xx errors instead of aborting long Colab runs
    for attempt in range(4):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            resp = getattr(exc, "response", None) or getattr(exc, "resp", None)
            code = getattr(resp, "status_code", None) or getattr(resp, "status", None)
            if attempt == 3 or code not in (429, 500, 502, 503, 504):
                raise
            time.sleep(2 ** (attempt + 1))

def _ensure_drive_folder(name, parent_id=None):
    key = (parent_id or "root", name)
    if key in _FOLDERS:
        return _FOLDERS[key]
    safe = name.replace("'", "\\'")
    parent = parent_id or "root"
    q = f"mimeType='application/vnd.google-apps.folder' and name='{safe}' and '{parent}' in parents and trashed=false"
    res = _gs_retry(lambda: DRIVE_SVC.files().list(q=q, fields="files(id)", pageSize=1).execute())
    files = res.get("files") or []
    if files:
        fid = files[0]["id"]
    else:
        meta = {"name": name, "mimeType": "application/vnd.google-apps.folder"}
        if parent_id:
            meta["parents"] = [parent_id]
        fid = _gs_retry(lambda: DRIVE_SVC.files().create(body=meta, fields="id").execute())["id"]
    _FOLDERS[key] = fid
    return fid

OUT_FOLDER_NAME = OUTPUT_DRIVE_FOLDER.strip() or f"Harmony4D_Benchmark_{REPORT_TAG}"
ROOT_OUT_FOLDER_ID = _ensure_drive_folder(OUT_FOLDER_NAME, DRIVE_FOLDER_ID.strip() or None)
if PUBLIC_READ:
    try:
        _gs_retry(lambda: DRIVE_SVC.permissions().create(fileId=ROOT_OUT_FOLDER_ID, body={"type": "anyone", "role": "reader"}).execute())
    except Exception:
        pass
print(f"[GDrive] Thư mục kết quả: {OUT_FOLDER_NAME}\n  📁 https://drive.google.com/drive/folders/{ROOT_OUT_FOLDER_ID}", flush=True)

def _seq_folder_id(seq_name):
    return _ensure_drive_folder(seq_name, ROOT_OUT_FOLDER_ID)

def _find_sheet_in_folder(title, folder_id):
    safe = title.replace("'", "\\'")
    q = f"mimeType='application/vnd.google-apps.spreadsheet' and name='{safe}' and '{folder_id}' in parents and trashed=false"
    res = _gs_retry(lambda: DRIVE_SVC.files().list(q=q, fields="files(id)", pageSize=1).execute())
    files = res.get("files") or []
    return _gs_retry(GC.open_by_key, files[0]["id"]) if files else None

def open_or_create_sheet(seq_name, title):
    if title in _SHEETS:
        return _SHEETS[title]
    folder_id = _seq_folder_id(seq_name)
    sh = _find_sheet_in_folder(title, folder_id)
    if sh is None:
        try:
            sh = _gs_retry(GC.create, title, folder_id=folder_id)
        except TypeError:
            sh = _gs_retry(GC.create, title)
            _gs_retry(lambda: DRIVE_SVC.files().update(fileId=sh.id, addParents=folder_id, removeParents="root", fields="id").execute())
        print(f"[GSheet] Tạo mới ({seq_name}/): {title} -> {sh.url}", flush=True)
    _SHEETS[title] = sh
    SHEET_URLS[title] = sh.url
    return sh

def _get_or_create_tab(sh, tab_name, n_rows, n_cols):
    try:
        return _gs_retry(sh.worksheet, tab_name)
    except gspread.WorksheetNotFound:
        first = sh.sheet1
        if tab_name == METHODS[0] and first.title in ("Sheet1", "Trang tính1"):
            _gs_retry(first.update_title, tab_name)
            return first
        return _gs_retry(sh.add_worksheet, title=tab_name, rows=max(100, n_rows + 20), cols=max(26, n_cols + 2))

def _write_tab(sh, cache_key, tab_name, rows):
    sig = (len(rows), tuple(rows[-1]) if rows else ())
    if _LAST_SYNCED.get(cache_key) == sig:
        return
    ncols = len(rows[0])
    ws = _get_or_create_tab(sh, tab_name, len(rows), ncols)
    if ws.row_count < len(rows) or ws.col_count < ncols:
        _gs_retry(ws.resize, rows=max(ws.row_count, len(rows) + 20), cols=max(ws.col_count, ncols + 2))
    _gs_retry(ws.clear)
    try:
        _gs_retry(ws.update, values=rows, range_name="A1")
    except TypeError:
        _gs_retry(ws.update, rows)
    decorate(ws, len(rows), ncols)  # freeze(rows=1) + header bold + zebra rows
    _LAST_SYNCED[cache_key] = sig

def write_person_sheet(seq_name, title, tab_name, header, person_rows, is_final=False):
    # ponytail: skip redundant API writes when row count/tail unchanged
    rows = [header, *person_rows]
    if is_final:
        rows.append(["End"] * len(header))
    sh = open_or_create_sheet(seq_name, title)
    _write_tab(sh, (title, tab_name, is_final), tab_name, rows)
    if is_final:
        print(f"[GSheet] Hoàn tất tab '{tab_name}' ({len(person_rows)} cặp): {seq_name}/{title}\n  🔗 {sh.url}", flush=True)

def _flt(v):
    try:
        x = float(str(v).strip().replace(",", "."))
        return x if x == x and abs(x) < 1e12 else None
    except (ValueError, TypeError):
        return None

def _pct_gain(ref, val):
    return round((ref - val) * 100.0 / ref, 2) if (ref is not None and val is not None and ref != 0) else "N/A"

def build_compare_rows(method_name, m_header, m_rows, n_header, n_rows):
    """Side-by-side comparison per pair: method vs baseline (mono) AND method vs naive."""
    def idx_map(h, rows):
        ci = {name: i for i, name in enumerate(h)}
        out = {}
        for r in rows:
            g = lambda k, row=r: row[ci[k]] if k in ci and ci[k] < len(row) else ""
            key = (g("Segment"), g("Target Person"), g("Cam Master"), g("Cam Slave"))
            if key[2] and key[3]:
                out[key] = g
        return out
    mm, nn = idx_map(m_header, m_rows), idx_map(n_header, n_rows)
    keys = sorted(set(mm) | set(nn))
    if not keys:
        return None
    metrics = [
        ("PA", "pa-mpjpe.all_baseline", "pa-mpjpe.all_method"),
        ("MP", "mpjpe.all_baseline", "mpjpe.all_method"),
        ("Acc", "accel.baseline", "accel.method"),
        ("MBLE", "mble.baseline", "mble.method"),
    ]
    header = ["Segment", "Target Person", "Cam Master", "Cam Slave", "Frames"]
    for tag, _, _ in metrics:
        header += [f"{tag}_baseline", f"{tag}_naive", f"{tag}_{method_name}", f"%d{tag}_vs_base", f"%d{tag}_vs_naive"]
    rows = [header]
    acc = [[] for _ in range(len(header) - 5)]
    for key in keys:
        gm, gn = mm.get(key), nn.get(key)
        frames = (gm("Evaluated Frames") if gm else "") or (gn("Evaluated Frames") if gn else "")
        row = [*key, frames]
        col = 0
        for _, b_col, m_col in metrics:
            vb = _flt(gm(b_col)) if gm else (_flt(gn(b_col)) if gn else None)
            vn = _flt(gn(m_col)) if gn else None
            vm = _flt(gm(m_col)) if gm else None
            vals = [
                round(vb, 2) if vb is not None else "N/A",
                round(vn, 2) if vn is not None else "N/A",
                round(vm, 2) if vm is not None else "N/A",
                _pct_gain(vb, vm),
                _pct_gain(vn, vm),
            ]
            for v in vals:
                if isinstance(v, (int, float)):
                    acc[col].append(v)
                col += 1
            row.extend(vals)
        rows.append(row)
    avg = ["AVERAGE", keys[0][1], f"{len(keys)} pairs", "", ""] + [
        round(sum(xs) / len(xs), 2) if xs else "N/A" for xs in acc
    ]
    rows.append(avg)
    return rows

def split_rows_by_person(report_path):
    if not report_path.is_file():
        return None, {}
    with report_path.open("r", encoding="utf-8-sig", newline="") as f:
        all_rows = list(csv.reader(f))
    if len(all_rows) < 2:
        return None, {}
    header = all_rows[0]
    tp_idx = header.index("Target Person") if "Target Person" in header else -1
    seg_idx = header.index("Segment") if "Segment" in header else -1
    proto_idx = header.index("Protocol") if "Protocol" in header else -1
    by_person = {}
    for row in all_rows[1:]:
        if not row or (proto_idx >= 0 and proto_idx < len(row) and row[proto_idx] == "End"):
            continue
        if tp_idx >= 0 and tp_idx < len(row) and row[tp_idx] not in ("", "End"):
            person = row[tp_idx].strip().lower()
        elif seg_idx >= 0 and seg_idx < len(row) and "_" in row[seg_idx]:
            person = row[seg_idx].rsplit("_", 1)[-1].strip().lower()
        else:
            person = "aria01"
        by_person.setdefault(person, []).append(row)
    return header, by_person

def restore_from_gsheets(seq_name, tab_name, report_path, candidates=("aria01", "aria02", "aria03", "aria04")):
    """Pull existing per-person Google Sheet tab from this sequence's Drive folder so benchmark resumes cleanly."""
    folder_id = _seq_folder_id(seq_name)
    header, combined = None, []
    for person in candidates:
        title = sheet_title(seq_name, person)
        try:
            sh = _SHEETS.get(title) or _find_sheet_in_folder(title, folder_id)
        except Exception as exc:
            print(f"[GSheet] Bỏ qua resume {title} do lỗi tạm thời: {exc}", flush=True)
            continue
        if sh is None:
            continue
        _SHEETS[title] = sh
        SHEET_URLS[title] = sh.url
        try:
            ws = _gs_retry(sh.worksheet, tab_name)
        except gspread.WorksheetNotFound:
            continue
        vals = _gs_retry(ws.get_all_values)
        if len(vals) < 2:
            continue
        if header is None:
            header = vals[0]
        p_idx = header.index("Protocol") if "Protocol" in header else -1
        rows = [r for r in vals[1:] if r and not (p_idx >= 0 and p_idx < len(r) and r[p_idx] == "End")]
        combined.extend(rows)
        print(f"[GSheet] Resume {len(rows)} cặp từ {seq_name}/{title} [{tab_name}]", flush=True)
    if header and combined:
        report_path.parent.mkdir(parents=True, exist_ok=True)
        with report_path.open("w", encoding="utf-8-sig", newline="") as f:
            csv.writer(f).writerows([header, *combined])

def sync_seq_to_gsheets(seq_name, tab_name, report_path, is_final=False):
    header, by_person = split_rows_by_person(report_path)
    if not header or not by_person:
        return
    for person, rows in sorted(by_person.items()):
        write_person_sheet(seq_name, sheet_title(seq_name, person), tab_name, header, rows, is_final=is_final)

def sync_compare_tabs(seq_name):
    if "naive" not in METHODS or FUSION_METHOD == "naive":
        return
    mh, m_by_p = split_rows_by_person(out_root / seq_name / METHOD_REPORTS[FUSION_METHOD][0])
    nh, n_by_p = split_rows_by_person(out_root / seq_name / METHOD_REPORTS["naive"][0])
    if not mh or not nh:
        return
    for person in sorted(set(m_by_p) | set(n_by_p)):
        comp = build_compare_rows(FUSION_METHOD, mh, m_by_p.get(person, []), nh, n_by_p.get(person, []))
        if comp:
            title = sheet_title(seq_name, person)
            sh = open_or_create_sheet(seq_name, title)
            _write_tab(sh, (title, "compare"), "compare", comp)
            print(f"[GSheet] Đã cập nhật tab 'compare' ({FUSION_METHOD} vs baseline & naive): {seq_name}/{title}", flush=True)

out_root = REPO_DIR / "output" / "drive_benchmark"
cpus = os.cpu_count() or 2
workers = WORKERS if WORKERS > 0 else (cpus if cpus <= 2 else cpus - 1)
targets = [REPO_DIR / c for c in CONFIGS.split()] if CONFIGS.strip() else sorted((REPO_DIR / "configs").glob("*.yml"))
targets = [p for p in targets if any(is_drive_segment(s) for s in (yaml.safe_load(p.read_text(encoding="utf-8")) or {}).get("segments", []))]
if REVERSE:
    targets = list(reversed(targets))

env = {**os.environ, "PYTHONUTF8": "1", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
for cfg_file in targets:
    seq_name = cfg_file.stem
    mounted_ok = DRIVE_ROOT is not None and stage_from_mounted_drive(cfg_file, seq_name)
    all_ok = True
    for method_name in METHODS:
        apply_method_config(method_name)
        work_report = out_root / seq_name / METHOD_REPORTS[method_name][0]
        restore_from_gsheets(seq_name, method_name, work_report)
        cmd = [sys.executable, "-u", "run_drive_benchmark.py", "--workers", str(workers), *(["--local-only"] if mounted_ok else []), *(["--reverse"] if REVERSE else []), str(cfg_file)]
        print(f"[{method_name}] $", " ".join(cmd), flush=True)
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        stop_sync = threading.Event()
        def _sync_loop(seq=seq_name, m=method_name, rp=work_report):
            while not stop_sync.wait(60):
                try:
                    sync_seq_to_gsheets(seq, m, rp, is_final=False)
                except Exception as exc:
                    print(f"[GSheet] Sync warning ({seq}/{m}): {exc}", flush=True)
        t = threading.Thread(target=_sync_loop, daemon=True)
        t.start()
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
        stop_sync.set()
        t.join()
        try:
            sync_seq_to_gsheets(seq_name, method_name, work_report, is_final=(proc.returncode == 0))
        except Exception as exc:
            print(f"[GSheet] Final sync warning ({seq_name}/{method_name}): {exc}", flush=True)
        if proc.returncode != 0:
            all_ok = False
        if proc.returncode not in (0, 1):
            raise RuntimeError(f"{cfg_file.name} ({method_name}) failed with exit code {proc.returncode}")
    try:
        sync_compare_tabs(seq_name)
    except Exception as exc:
        print(f"[GSheet] Compare tab warning ({seq_name}): {exc}", flush=True)
    if all_ok and mounted_ok:
        shutil.rmtree(REPO_DIR / "input" / "drive_benchmark" / seq_name, ignore_errors=True)

In [ ]:
#@title 3. Tổng hợp kết quả trực tiếp từ Google Sheets & kiểm toán Person Mapping
import glob, json
from pathlib import Path
import pandas as pd

def _num(s):
    return pd.to_numeric(s, errors="coerce")

rows = []
for title, url in sorted(SHEET_URLS.items()):
    sh = _SHEETS.get(title) or GC.open(title)
    def _tab_df(name):
        try:
            vals = _gs_retry(sh.worksheet(name).get_all_values)
        except Exception:
            return pd.DataFrame()
        if len(vals) < 2:
            return pd.DataFrame()
        df = pd.DataFrame(vals[1:], columns=vals[0])
        return df[df["Protocol"].astype(str) != "End"] if "Protocol" in df else df
    df_m = _tab_df(FUSION_METHOD)
    if df_m.empty:
        continue
    df_n = _tab_df("naive") if ("naive" in METHODS and FUSION_METHOD != "naive") else pd.DataFrame()
    for (seg, person), g in df_m.groupby(["Segment", "Target Person"]):
        pa_b, pa_m = _num(g["pa-mpjpe.all_baseline"]).mean(), _num(g["pa-mpjpe.all_method"]).mean()
        mp_b, mp_m = _num(g["mpjpe.all_baseline"]).mean(), _num(g["mpjpe.all_method"]).mean()
        acc_b = _num(g["accel.baseline"]).mean() if "accel.baseline" in g else None
        acc_m = _num(g["accel.method"]).mean() if "accel.method" in g else None
        gn = df_n[(df_n["Segment"] == seg) & (df_n["Target Person"] == person)] if not df_n.empty else pd.DataFrame()
        pa_n = _num(gn["pa-mpjpe.all_method"]).mean() if not gn.empty else None
        mp_n = _num(gn["mpjpe.all_method"]).mean() if not gn.empty else None
        acc_n = _num(gn["accel.method"]).mean() if (not gn.empty and "accel.method" in gn) else None
        rows.append({
            "Google Sheet": title, "Segment": seg, "Person": person, "Pairs": len(g),
            "PA base": round(pa_b, 2), "PA naive": round(pa_n, 2) if pa_n is not None else None, f"PA {FUSION_METHOD}": round(pa_m, 2),
            "dPA vs base": round(pa_m - pa_b, 2), "dPA vs naive": round(pa_m - pa_n, 2) if pa_n is not None else None,
            "MP base": round(mp_b, 2), "MP naive": round(mp_n, 2) if mp_n is not None else None, f"MP {FUSION_METHOD}": round(mp_m, 2),
            "dMP vs base": round(mp_m - mp_b, 2), "dMP vs naive": round(mp_m - mp_n, 2) if mp_n is not None else None,
            "Acc base": round(acc_b, 2) if acc_b is not None else None,
            "Acc naive": round(acc_n, 2) if acc_n is not None else None,
            f"Acc {FUSION_METHOD}": round(acc_m, 2) if acc_m is not None else None,
            "URL": url,
        })
display(pd.DataFrame(rows))

map_rows = []
for path in sorted(glob.glob(str(REPO_DIR / "output/drive_benchmark/*/person_labels_report.json"))):
    rep = json.loads(Path(path).read_text(encoding="utf-8"))
    seq = Path(path).parent.name
    for cam_id, info in sorted((rep.get("cameras") or {}).items()):
        map_rows.append({
            "Sequence": seq, "Camera": cam_id,
            "Mapping": info.get("mapping"),
            "Covered": info.get("frames_covered"),
            "Switch frames": info.get("switch_frames"),
        })
if map_rows:
    print("\n=== Person Mapping Audit ===")
    display(pd.DataFrame(map_rows))